# KMDigits reusable template — k-means on any numeric matrix

**Short name:** `KMDigits`

Swap the loader. Keep the rest: scale if units differ, pick k (domain or elbow), fit, map clusters to any available labels, inspect centroids, simulate knobs.

Works for: digits, iris, customer RFM, color quantization, country-year macro panels.


## Checklist

1. Rows = samples, columns = numeric features. Drop ids and raw labels from `X`.
2. If columns live on different scales, `StandardScaler` (fit on the matrix you cluster).
3. Choose `k` from the story first, elbow second.
4. `KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)`.
5. If labels exist, majority-map + purity + ARI. If not, stop at inertia + centroid plots.
6. Predict new rows with the **same** columns, **same** scaler.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from KMDigits import map_clusters_to_truth

# --- 1. loader (edit this block) ---------------------------------
df = pd.read_csv("data/digits.csv")          # or iris.csv, or your file
label_col = "digit"                          # set None if unlabeled
feature_cols = [c for c in df.columns if c != label_col]
X_raw = df[feature_cols].to_numpy(dtype=float)
y = None if label_col is None else df[label_col].to_numpy()

# --- 2. optional scale -------------------------------------------
SCALE = False                                # True when units mix
if SCALE:
    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)
else:
    scaler = None
    X = X_raw

# --- 3. k --------------------------------------------------------
k = 10
model = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
print("inertia", round(model.inertia_, 2), "sizes", np.bincount(model.labels_))

if y is not None:
    mapped, mapping, purity = map_clusters_to_truth(model.labels_, y, k)
    print("purity", round(purity, 4), "ARI", round(adjusted_rand_score(y, model.labels_), 4))
    print("map", mapping)

# --- 4. elbow ----------------------------------------------------
ks = range(1, min(16, len(X)))
inert = [KMeans(n_clusters=kk, n_init=3, random_state=0).fit(X).inertia_ for kk in ks]
plt.plot(list(ks), inert, marker="o")
plt.xlabel("k"); plt.ylabel("inertia"); plt.title("elbow")
plt.show()

# --- 5. 2-D view -------------------------------------------------
Z = PCA(2, random_state=0).fit_transform(X)
plt.scatter(Z[:, 0], Z[:, 1], c=model.labels_, s=10, cmap="tab10")
plt.title("PCA colored by cluster"); plt.show()

# --- 6. infer one new row ----------------------------------------
# new = scaler.transform(new_raw) if scaler else new_raw
# print(model.predict(new))


## Simulation stub

Change `k`, `n`, `noise` in a small grid and record inertia / purity. See §14 of the solution notebook.
